# Sparse Walker: temporal skip memories on ML-1M

K=8, degree=4 and two graph hops remain unchanged. We add three frozen-in-time sparse landmarks (short / medium / long) and warm-start from the best plain ML-1M Walker checkpoint.

**Run the next cell first (or use Runtime → Run all).** It is self-bootstrapping and safe to rerun.

In [ ]:
# SELF-BOOTSTRAPPING SETUP + SMOKE TEST
import os, sys, subprocess, pathlib
REPO='/content/Sparsewalker'
BRANCH='agent/walker-temporal-skips'
if not pathlib.Path(REPO, '.git').exists():
    subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO], check=True)
else:
    subprocess.run(['git','-C',REPO,'fetch','-q','origin',BRANCH], check=True)
    subprocess.run(['git','-C',REPO,'checkout','-q',BRANCH], check=True)
    subprocess.run(['git','-C',REPO,'reset','--hard',f'origin/{BRANCH}'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',REPO], check=True)
SRC=f'{REPO}/src'
if SRC not in sys.path: sys.path.insert(0,SRC)
os.chdir(REPO)

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import torch, sparsewalker
from sparsewalker.models import SparseWalkerTemporalMemory
print('sparsewalker from', sparsewalker.__file__)
print('torch', torch.__version__)
print('GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print('bf16', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

m=SparseWalkerTemporalMemory(3706,200,d=64,layers=2,side=256,h=16,active=8,top_side=2,degree=4).cuda()
x=torch.randint(1,3707,(4,40),device='cuda')
with torch.autocast('cuda',dtype=torch.bfloat16):
    H=m(x)
    loss=H.float().square().mean()
loss.backward()
print('SMOKE OK', H.shape, 'loss', float(loss))
del m,x,H,loss
torch.cuda.empty_cache()

## Run the temporal-skip diagnostic

Uses the existing base checkpoint at `/content/drive/MyDrive/sparsewalker_canonical_pair/ml1m/seed42/SparseWalker_FullCE/best.pt`. Results are written separately to `sparsewalker_temporal_skips`.

In [ ]:
import os, pathlib, subprocess, sys
assert pathlib.Path('/content/Sparsewalker/experiments/run_ml1m_temporal_skips.py').exists(), 'Run the setup/smoke cell above first.'
os.chdir('/content/Sparsewalker')
cmd=[sys.executable,'experiments/run_ml1m_temporal_skips.py','--seed','42','--max-epochs','20','--eval-every','2','--patience','8','--batch-size','128','--eval-batch-size','1024']
subprocess.run(cmd, check=True)

## Inspect the learning curve

In [ ]:
import pandas as pd
from pathlib import Path
p=Path('/content/drive/MyDrive/sparsewalker_temporal_skips/ml1m/seed42/history.csv')
if p.exists():
    df=pd.read_csv(p)
    display(df[['epoch','loss','NDCG@10','HR@10','MRR@10','memory_share','seconds','positions_per_s']])
    print('best temporal val NDCG@10', df['NDCG@10'].max())
else:
    print('No history yet')